# 03 — Lab Measurements & Genotypic Extraction
## NIH All of Us — Sickle Cell Disease (SCD) Analysis

| Field | Details |
|-------|---------|
| **Project** | Sickle Cell Disease Multi-Modal Analysis |
| **Dataset** | All of Us Controlled Tier Dataset v8 |
| **Description** | Extracts laboratory measurements (CBC, HbF, LDH, etc.) from the BigQuery measurement table, performs Mann-Whitney U statistical tests comparing lab values across complication severity groups, and runs KMeans clustering for patient stratification |
| **Input** | `SCD_Demographics.csv`, `SCD_Complication_Severity_Only.csv`, All of Us CDR BigQuery (`measurement`) |
| **Output** | `SCD_Lab_Measurements_Long.csv`, `lab_measurements_wide_with_units.csv`, `ml_ready_latest.csv`, `out_complications/complication_lab_mwu_results.csv` |

---

## Workflow
1. Extract lab measurements for 476 SCD participants from BigQuery  
2. Pivot to wide format (one row per patient, one column per lab test)  
3. Compute cohort-level summary statistics and distributions  
4. Mann-Whitney U tests: lab values vs complication severity groups  
5. KMeans clustering for patient stratification  
6. Visualise results with heatmaps and bar charts

---
> ⚠️ **All of Us Compliance:** This notebook must be run inside the All of Us Researcher Workbench. No participant data is stored in this repository.

---
## ⚙️ Before You Begin

> **Prerequisite:** You must have run both `01_EHR_demographics_extraction.ipynb`
> and `02_phenotypic_complications_extraction.ipynb` first.  
> This notebook requires `SCD_Demographics.csv` and `SCD_Complication_Severity_Only.csv`
> to already exist in your workspace.

### What this notebook does
- Extracts laboratory measurements from All of Us BigQuery `measurement` table  
- Pivots data to wide format (one row per patient, one column per lab test)  
- Computes cohort-level summary statistics and distributions  
- Runs **Mann-Whitney U tests** comparing lab values across complication severity groups  
- Performs **KMeans clustering** (k=3) for patient stratification  
- Visualises results using heatmaps, bar charts and PCA scatter plots  

### Run order
1. Select **Python 3** kernel  
2. Click **Kernel → Restart & Run All**  
   *OR* run cells one by one with **Shift + Enter**

---

## Step 1 — Extract Lab Measurements from BigQuery

**What this does:**  
Loads the list of 476 SCD participants from `SCD_Demographics.csv`,
then queries the All of Us BigQuery `measurement` table to pull all
lab results recorded for these participants.

**Lab tests extracted include:**
- Haemoglobin (Hb)
- White Blood Cell count (WBC)
- Platelet count
- Lactate Dehydrogenase (LDH)
- Foetal Haemoglobin (HbF)
- Reticulocyte count
- Bilirubin (total)
- Creatinine

> ⏱️ *This BigQuery query may take 3–7 minutes due to the measurement table size.*

In [ ]:
import pandas as pd

# Load measurement data from exported file (or directly from memory if already exists)
# measurement_df = pd.read_csv("measurement.csv")  # if you exported to CSV
# OR if it's already there from your export, skip loading

# Convert date to datetime if needed
measurement_df["measurement_date"] = pd.to_datetime(measurement_df["measurement_date"])

# Filter for labs of interest (optional)
lab_concepts = [
    40795725, 3018738, 37045413, 40789184, 40782735,
    40789179, 3000905, 303751, 3013869, 3010451,
    3011948, 3008342, 37060058, 37073425, 3004249,
    3012888, 3027018, 40775801
]

filtered = measurement_df[measurement_df["measurement_concept_id"].isin(lab_concepts)]

# Get most recent lab per person per test
most_recent_labs = (
    filtered.sort_values(["person_id", "measurement_concept_id", "measurement_date"], ascending=[True, True, False])
    .drop_duplicates(subset=["person_id", "measurement_concept_id"])
    .reset_index(drop=True)
)

# Optional: map readable test names
lab_map = {
    40795725: "Hemoglobin",
    3018738: "Fetal hemoglobin",
    37045413: "Red blood cell count",
    40789184: "Mean corpuscular volume",
    40782735: "Mean corpuscular hemoglobin",
    40789179: "Hematocrit",
    3000905: "WBC (Leukocytes)",
    303751: "Lymphocytes",
    3013869: "Basophils",
    3010451: "Eosinophils",
    3011948: "Monocytes",
    3008342: "Neutrophils",
    37060058: "Reticulocyte count",
    37073425: "Platelet count",
    3004249: "Systolic blood pressure",
    3012888: "Diastolic blood pressure",
    3027018: "Heart rate",
    40775801: "Creatinine"
}

most_recent_labs["test_name"] = most_recent_labs["measurement_concept_id"].map(lab_map)

# Show result
print(most_recent_labs.head(10))


In [ ]:
import os
import pandas as pd

# Step 1: Load the 476-person demographic file
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df['person_id'].astype(str).unique().tolist()

# Step 2: Prepare SQL IN clause (chunk if needed to avoid query size limits)
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])  # wrap in quotes if string-type IDs

# Step 3: Lab concept IDs
lab_concept_ids = [
    40795725, 3018738, 37045413, 40789184, 40782735, 40789179,
    3000905, 303751, 3013869, 3010451, 3011948, 3008342,
    37060058, 37073425, 3004249, 3012888, 3027018, 40775801
]
concept_ids_str = ", ".join([str(cid) for cid in lab_concept_ids])

# Step 4: Query measurements for those patients
lab_measurements_df = pd.read_gbq(
    f"""
    SELECT
        m.person_id,
        m.measurement_concept_id,
        m.measurement_date,
        m.value_as_number,
        m.unit_concept_id,
        m.unit_source_value
    FROM
        `{os.environ["WORKSPACE_CDR"]}.measurement` m
    WHERE
        CAST(m.person_id AS STRING) IN ({formatted_ids})
        AND m.measurement_concept_id IN ({concept_ids_str})
    """,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

# Step 5: Add readable test names
lab_map = {
    40795725: "Hemoglobin",
    3018738: "Fetal hemoglobin",
    37045413: "Red blood cell count",
    40789184: "Mean corpuscular volume",
    40782735: "Mean corpuscular hemoglobin",
    40789179: "Hematocrit",
    3000905: "WBC ",
    303751: "Lymphocytes",
    3013869: "Basophils",
    3010451: "Eosinophils",
    3011948: "Monocytes",
    3008342: "Neutrophils",
    37060058: "Reticulocyte count",
    40779159: "Platelets",
    3004249: "Systolic blood pressure",
    3012888: "Diastolic blood pressure",
    3027018: "Heart rate",
    40775801: "Creatinine"
}
lab_measurements_df["lab_test"] = lab_measurements_df["measurement_concept_id"].map(lab_map)

# Step 6: Preview results
lab_measurements_df.head(10)




In [ ]:
# Step 6: Preview results
lab_measurements_df.head(10)

# Step 7: Save to CSV
lab_measurements_df.to_csv("SCD_Lab_Measurements.csv", index=False)
print("✅ File saved as 'SCD_Lab_Measurements.csv'")


In [ ]:
import os
import pandas as pd

# Step 1: Load the list of 476 patients
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df['person_id'].astype(str).unique().tolist()

# Step 2: Split into manageable chunks if needed (Google has limits)
# But for 476 patients, one query is fine
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])

# Step 3: Lab test concept IDs
lab_map = {
    3000963: "Hemoglobin",
    3018738: "Fetal hemoglobin",
    37045413: "Red blood cell count",
    40789184: "Mean corpuscular volume",
    40782735: "Mean corpuscular hemoglobin",
    40789179: "Hematocrit",
    3000905: "WBC ",
    303751: "Lymphocytes",
    3013869: "Basophils",
    3010451: "Eosinophils",
    3011948: "Monocytes",
    3008342: "Neutrophils",
    37060058: "Reticulocyte count",
    40779159: "Platelets",
    3004249: "Systolic blood pressure",
    3012888: "Diastolic blood pressure",
    3027018: "Heart rate",
    3016723: "Creatinine"
}
concept_ids_str = ", ".join([str(cid) for cid in lab_map.keys()])

# Step 4: Query measurement table for your 476 patients and selected lab tests
lab_df = pd.read_gbq(
    f"""
    SELECT
        person_id,
        measurement_concept_id,
        measurement_date,
        value_as_number
    FROM
        `{os.environ["WORKSPACE_CDR"]}.measurement`
    WHERE
        CAST(person_id AS STRING) IN ({formatted_ids})
        AND measurement_concept_id IN ({concept_ids_str})
    """,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

# Step 5: Add readable names
lab_df["lab_test"] = lab_df["measurement_concept_id"].map(lab_map)

# Step 6: Keep only the latest measurement per patient per test
lab_df = lab_df.sort_values(by=["person_id", "lab_test", "measurement_date"])
latest_labs = lab_df.groupby(["person_id", "lab_test"]).tail(1)

# Step 7: Pivot to wide format for ML
ml_ready = latest_labs.pivot(index="person_id", columns="lab_test", values="value_as_number").reset_index()

# Optional: Fill missing values (if a patient is missing a lab)
#ml_ready.fillna(np.nan, inplace=True)  # or use mean/median imputation later

# Step 8: Final preview
print("✅ Final ML-ready lab dataset shape:", ml_ready.shape)
ml_ready.head(10)


In [ ]:
ml_ready.info()

---
## Step 2 — Save Lab Data in Long & Wide Format

**What this does:**  
Saves two versions of the lab data:
- **Long format** (`SCD_Lab_Measurements_Long.csv`) — one row per lab result per participant (with date and concept_id)
- **Wide format** (`lab_measurements_wide_with_units.csv`) — one row per participant, one column per lab test

The wide format is used for all downstream statistical analysis and ML.

In [ ]:
import os
import pandas as pd

# Step 1: Load the 476-person demographic file
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df['person_id'].astype(str).unique().tolist()

# Step 2: Prepare SQL IN clause (chunk if needed to avoid query size limits)
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])  # wrap in quotes if string-type IDs

# Step 3: Lab concept IDs
lab_concept_ids = [
    40795725, 3018738, 37045413, 40789184, 40782735, 40789179,
    3000905, 303751, 3013869, 3010451, 3011948, 3008342,
    37060058, 37073425, 3004249, 3012888, 3027018, 40775801
]
concept_ids_str = ", ".join([str(cid) for cid in lab_concept_ids])

# Step 4: Query measurements for those patients
lab_measurements_df = pd.read_gbq(
    f"""
    SELECT
        m.person_id,
        m.measurement_concept_id,
        m.measurement_date,
        m.value_as_number,
        m.unit_concept_id,
        m.unit_source_value
    FROM
        `{os.environ["WORKSPACE_CDR"]}.measurement` m
    WHERE
        CAST(m.person_id AS STRING) IN ({formatted_ids})
        AND m.measurement_concept_id IN ({concept_ids_str})
    """,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

# Step 5: Add readable test names
lab_map = {
    3000963: "Hemoglobin",
    3018738: "Fetal hemoglobin",
    37045413: "Red blood cell count",
    40789184: "Mean corpuscular volume",
    40782735: "Mean corpuscular hemoglobin",
    40789179: "Hematocrit",
    3000905: "WBC (Leukocytes)",
    303751: "Lymphocytes",
    3013869: "Basophils",
    3010451: "Eosinophils",
    3011948: "Monocytes",
    3008342: "Neutrophils",
    37060058: "Reticulocyte count",
    37073425: "Platelet count",
    3004249: "Systolic blood pressure",
    3012888: "Diastolic blood pressure",
    3027018: "Heart rate",
    3016723: "Creatinine"
}
lab_measurements_df["lab_test"] = lab_measurements_df["measurement_concept_id"].map(lab_map)

# Step 6: Preview results
lab_measurements_df.head(10)


In [ ]:
# ✅ Save long format: all lab rows per patient (with date + concept_id)
lab_df.to_csv("SCD_Lab_Measurements_Long.csv", index=False)
print("✅ Saved long format to SCD_Lab_Measurements_Long.csv")

# ✅ Save wide format: one row per patient with each lab test as a column
ml_ready.to_csv("SCD_Lab_Measurements_Wide.csv", index=False)
print("✅ Saved wide format to SCD_Lab_Measurements_Wide.csv")


---
## Step 3 — Visualise Lab Value Distributions

**What this does:**  
Plots distribution curves for each lab test across the cohort,
overlaid with a normal distribution to identify skewness or outliers.

**Expected output:** One chart per lab analyte showing the distribution shape.

In [ ]:
print("Total participants:", df.shape[0])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Load wide-format lab data
df = pd.read_csv("lab_measurements_wide_with_units.csv")

# Exclude non-lab columns
lab_columns = [col for col in df.columns if col != "person_id"]

# Loop through each lab measurement
for lab in lab_columns:
    values = df[lab].dropna()

    if len(values) < 10:
        print(f"⚠️ Not enough data for {lab}, skipping...")
        continue

    # Compute median
    median_val = np.median(values)

    # Fit normal curve
    mu, std = norm.fit(values)

    # Plot
    plt.figure(figsize=(8, 5))
    n, bins, patches = plt.hist(values, bins=20, density=True, alpha=0.6, edgecolor='black', label='Value distribution')

    x = np.linspace(values.min(), values.max(), 100)
    p = norm.pdf(x, mu, std)
    plt.plot(x, p, 'k', linewidth=2, label=f'Normal fit\nμ={mu:.2f}, σ={std:.2f}')

    # Plot median line
    plt.axvline(median_val, color='red', linestyle='--', label=f'Median = {median_val:.2f}')

    # Final plot settings
    plt.title(f'{lab} Distribution with Normal Fit')
    plt.xlabel(f'{lab}')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


---
## Step 4 — Summary Statistics for Lab Values

**What this does:**  
Computes median, mean, standard deviation and missingness rate
for each lab analyte across all 476 participants.

**Expected output:** A summary statistics table and a median bar chart.

In [ ]:
import pandas as pd

# Load wide-format lab data
df = pd.read_csv("lab_measurements_wide_with_units.csv")

# Drop person_id (it's not a lab test)
lab_only = df.drop(columns=["person_id"], errors='ignore')

# Calculate median for each lab column
lab_medians = lab_only.median(numeric_only=True).sort_values(ascending=False).reset_index()
lab_medians.columns = ["Lab Test", "Median Value"]

# Show top rows
print("✅ Median lab values across 476 SCD participants:")
print(lab_medians)

# Optionally save to CSV
lab_medians.to_csv("lab_measurement_medians.csv", index=False)


---
## Step 5 — Mann-Whitney U Test: Lab Values vs Complication Severity

**What this does:**  
For each lab test, compares values between participants in different
severity groups (None / Mild / Moderate / Severe) using the
**Mann-Whitney U test** (non-parametric, appropriate for non-normal lab data).

**Interpretation:**
- p < 0.05 → statistically significant difference between groups
- Results saved to `complication_lab_mwu_results.csv`

> This is the key statistical analysis linking lab values to disease severity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Load lab data
df = pd.read_csv("lab_measurements_wide_with_units.csv")

# Define lab tests to include (or use all columns except person_id)
lab_tests = [col for col in df.columns if col != "person_id"]

# Set up the grid size for subplots
n = len(lab_tests)
cols = 3
rows = (n + cols - 1) // cols

# Create figure and axes
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
axes = axes.flatten()

# Loop through each lab test
for i, lab in enumerate(lab_tests):
    ax = axes[i]
    values = df[lab].dropna()

    if len(values) < 10:
        ax.set_title(f"{lab} (Insufficient data)")
        ax.axis('off')
        continue

    median_val = np.median(values)
    mu, std = norm.fit(values)

    # Histogram
    ax.hist(values, bins=20, density=True, alpha=0.6, edgecolor='black')
    x = np.linspace(values.min(), values.max(), 100)
    p = norm.pdf(x, mu, std)
    ax.plot(x, p, 'k', linewidth=1.5)

    ax.axvline(median_val, color='red', linestyle='--')
    ax.set_title(f"{lab}\nμ={mu:.2f}, σ={std:.2f}, median={median_val:.2f}")
    ax.set_ylabel("Density")
    ax.set_xlabel("Value")

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle("Lab Measurement Distributions (SCD Participants)", fontsize=16, y=1.02)
plt.subplots_adjust(top=0.93)
plt.savefig("combined_lab_distributions.png", dpi=300)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# Load datasets
comp_df = pd.read_csv("SCD_Complication_Severity_Only.csv")
lab_df = pd.read_csv("lab_measurements_wide_with_units.csv")

# Convert binary complications to 'Yes'/'No'
comp_columns = [col for col in comp_df.columns if col not in ['person_id', 'severity_score', 'severity_group']]
comp_df[comp_columns] = comp_df[comp_columns].replace({1: 'Yes', 0: 'No'})

# Merge complication and lab data
merged_df = pd.merge(comp_df, lab_df, on="person_id", how="inner")

# Prepare results list
results = []

# Loop over each complication and each lab
lab_columns = [col for col in lab_df.columns if col != "person_id"]

for complication in comp_columns:
    for lab in lab_columns:
        temp = merged_df[[complication, lab, "severity_score"]].dropna()
        if temp[complication].nunique() != 2:
            continue

        group_yes = temp[temp[complication] == "Yes"][lab]
        group_no = temp[temp[complication] == "No"][lab]

        if len(group_yes) < 5 or len(group_no) < 5:
            continue

        # Mann-Whitney U test (non-parametric)
        stat, p = mannwhitneyu(group_yes, group_no, alternative='two-sided')

        # Severity comparison
        sev_yes = temp[temp[complication] == "Yes"]["severity_score"].median()
        sev_no = temp[temp[complication] == "No"]["severity_score"].median()

        results.append({
            "Complication": complication,
            "Lab Test": lab,
            "Median (Yes)": np.median(group_yes),
            "Median (No)": np.median(group_no),
            "Severity (Yes)": sev_yes,
            "Severity (No)": sev_no,
            "p-value": p
        })

# Create and sort results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="p-value")

# Save to CSV
results_df.to_csv("complication_lab_severity_stats.csv", index=False)

# Show top results
print("✅ Top complication-lab associations with severity and p-values:")
print(results_df.head(10))


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# === Load your files ===
comp_df = pd.read_csv("SCD_Complication_Severity_Only.csv")   # Must contain person_id, binary complications, severity_score
lab_df = pd.read_csv("lab_measurements_wide_with_units.csv")      # Must contain person_id and recent lab parameters

# === Keep only your 476 participants ===
scd_demo = pd.read_csv("SCD_Demographics.csv")
scd_ids = scd_demo["person_id"].unique()

comp_df = comp_df[comp_df["person_id"].isin(scd_ids)]
lab_df = lab_df[lab_df["person_id"].isin(scd_ids)]

# === Convert 0/1 to Yes/No for all complications ===
comp_columns = [col for col in comp_df.columns if col not in ["person_id", "severity_score", "severity_group"]]
comp_df[comp_columns] = comp_df[comp_columns].replace({1: "Yes", 0: "No"})

# === Merge lab and complication data ===
merged_df = pd.merge(comp_df, lab_df, on="person_id", how="inner")

# === Prepare output ===
results = []
lab_columns = [col for col in lab_df.columns if col != "person_id"]

# === Loop through each complication and lab test ===
for complication in comp_columns:
    for lab in lab_columns:
        temp = merged_df[[complication, lab, "severity_score"]].dropna()

        if temp[complication].nunique() != 2:
            continue  # must have both Yes and No

        group_yes = temp[temp[complication] == "Yes"][lab]
        group_no = temp[temp[complication] == "No"][lab]

        if len(group_yes) < 5 or len(group_no) < 5:
            continue  # too few values

        # Mann-Whitney U Test
        stat, p = mannwhitneyu(group_yes, group_no, alternative='two-sided')

        # Severity score medians
        sev_yes = temp[temp[complication] == "Yes"]["severity_score"].median()
        sev_no = temp[temp[complication] == "No"]["severity_score"].median()

        results.append({
            "Complication": complication,
            "Lab Test": lab,
            "Median (Yes)": np.median(group_yes),
            "Median (No)": np.median(group_no),
            "Severity (Yes)": sev_yes,
            "Severity (No)": sev_no,
            "p-value": p
        })

# === Format output ===
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("p-value")
results_df.to_csv("SCD_Complication_Lab_Severity_Stats.csv", index=False)

# === Show summary ===
print("✅ Analysis complete for 476 participants.")
print(results_df.head(10))


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# Step 1: Load data
labs = pd.read_csv("lab_measurements_wide_with_units.csv")
complications = pd.read_csv("SCD_Complication_Severity_Assessed.csv")

# Step 2: Merge both datasets on person_id
merged = pd.merge(complications, labs, on="person_id", how="inner")

# Step 3: Extract lab measurement column names
lab_columns = labs.columns.drop("person_id")

# Step 4: Extract complication flags (excluding person_id and severity_score)
complication_columns = [col for col in complications.columns if col not in ["person_id", "severity_score"]]

# Step 5: Prepare result storage
results = []

for comp in complication_columns:
    # Check unique values
    unique_vals = merged[comp].dropna().unique()
    if set(unique_vals) != {0, 1}:
        continue  # skip non-binary complication columns

    for lab in lab_columns:
        group_yes = merged[merged[comp] == 1][lab].dropna()
        group_no = merged[merged[comp] == 0][lab].dropna()

        if len(group_yes) < 5 or len(group_no) < 5:
            continue  # skip if not enough data

        # Mann-Whitney U Test
        try:
            stat, pval = mannwhitneyu(group_yes, group_no, alternative='two-sided')
        except Exception as e:
            pval = np.nan

        # Severity comparison
        severity_yes = merged.loc[merged[comp] == 1, "severity_score"].median()
        severity_no = merged.loc[merged[comp] == 0, "severity_score"].median()

        results.append({
            "Complication": comp,
            "Lab Test": lab,
            "Median (Yes)": group_yes.median(),
            "Median (No)": group_no.median(),
            "Severity (Yes)": severity_yes,
            "Severity (No)": severity_no,
            "p-value": pval
        })

# Step 6: Create final DataFrame
result_df = pd.DataFrame(results)
result_df = result_df.sort_values(by="p-value")

# Step 7: Save
result_df.to_csv("complication_lab_analysis.csv", index=False)

# Optional preview
print("✅ Analysis complete.")
print(result_df.head(10))


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# Step 1: Load data
labs = pd.read_csv("lab_measurements_wide_with_units.csv")  # wide lab data for 476 participants
complications = pd.read_csv("SCD_Complication_Severity_Assessed.csv")  # includes 0/1 columns for each complication

# Step 2: Merge datasets
merged = pd.merge(complications, labs, on="person_id", how="inner")

# Step 3: Define complications and lab columns
complication_cols = [col for col in complications.columns if col != "person_id" and complications[col].isin([0, 1]).any()]
lab_columns = [col for col in labs.columns if col != "person_id"]

# Step 4: Output folder
output_dir = "complication_lab_stats"
os.makedirs(output_dir, exist_ok=True)

# Step 5: Run analysis for each complication
summary_all = []

for comp in complication_cols:
    group_yes = merged[merged[comp] == 1]
    group_no = merged[merged[comp] == 0]

    results = []

    for lab in lab_columns:
        values_yes = group_yes[lab].dropna()
        values_no = group_no[lab].dropna()

        if len(values_yes) < 5 or len(values_no) < 5:
            continue

        try:
            stat, pval = mannwhitneyu(values_yes, values_no, alternative='two-sided')
        except Exception:
            pval = np.nan

        results.append({
            "Complication": comp,
            "Lab Test": lab,
            "Median (Yes)": np.median(values_yes),
            "Median (No)": np.median(values_no),
            "p-value": pval,
            "N (Yes)": len(values_yes),
            "N (No)": len(values_no)
        })

    if results:
        df = pd.DataFrame(results).sort_values("p-value")
        df.to_csv(f"{output_dir}/{comp}_lab_comparison.csv", index=False)
        summary_all.extend(df.to_dict(orient="records"))

# Step 6: Combine all results
final_summary = pd.DataFrame(summary_all)

# Step 7: Format for display
final_summary['p-value'] = final_summary['p-value'].apply(lambda p: f"{p:.4f}" if pd.notnull(p) else "NA")

# Step 8: Save and show
final_summary.to_csv("combined_lab_complication_summary.csv", index=False)

# Optional: Show top results in table
print("✅ Summary of all complications and lab comparisons:")
display(final_summary.head(20))  # You can change to .head(50) or remove for full table



In [ ]:
print(severity_df.columns.tolist())  # Check for typos in 'severity_group'
print(severity_df['severity_group'].unique())  # See what categories are present


---
## Step 6 — Extended Complication Analysis (Beyond Top 10)

**What this does:**  
Extends the MWU analysis to include all SCD complications (not just top 10)
by loading severity data for participants with zero complications
and querying remaining condition data from BigQuery.

In [ ]:
# Load full list of all complications
all_comps = pd.read_csv("scd_complications_concept_ids.csv")
all_comps = all_comps.rename(columns={"concept_id": "condition_concept_id"})

# Load top 10 complication concept IDs
top10_concept_ids = [
    137829, 4159748, 4322024, 4266931, 4064161,
    141663, 439393, 4021365, 315276, 4131942
]

# Filter to only non-top-10 complications
other_comps = all_comps[~all_comps["condition_concept_id"].isin(top10_concept_ids)]
other_comp_ids = other_comps["condition_concept_id"].tolist()


In [ ]:
import pandas as pd
from google.cloud import bigquery

# Step 1: Load severity data and filter to 'None' group
severity_df = pd.read_csv("SCD_Complication_Severity_Categorised.csv", dtype={"person_id": str})
severity_df["severity_group"] = severity_df["severity_group"].fillna("None")
none_ids = severity_df[severity_df["severity_group"] == "None"]["person_id"].tolist()

# Step 2: Load all complication concepts and filter out top 10
comp_df = pd.read_csv("scd_complications_concept_ids.csv")
comp_df = comp_df.rename(columns={"concept_id": "condition_concept_id"})

# List of top 10 concept IDs to exclude
top10_concept_ids = [
    137829, 4159748, 4322024, 4266931, 4064161,
    141663, 439393, 4021365, 315276, 4131942
]

# Get other complication concept IDs (excluding top 10)
other_comps = comp_df[~comp_df["condition_concept_id"].isin(top10_concept_ids)]
other_comp_ids = other_comps["condition_concept_id"].tolist()

# Step 3: Query BigQuery for other complications among 'None' participants
client = bigquery.Client()

# Convert person_ids to integers (if not already)
person_list = ",".join(str(int(pid)) for pid in none_ids[:1000])  # Safe batch limit
concept_list = ",".join(str(int(cid)) for cid in other_comp_ids)

# Final query with correct types
query = f"""
SELECT person_id, condition_concept_id
FROM `fc-aou-cdr-prod-ct.C2024Q3R5.condition_occurrence`
WHERE person_id IN ({person_list})
  AND condition_concept_id IN ({concept_list})
"""

condition_df = client.query(query).to_dataframe()

# Step 4: Merge with complication names
merged_df = condition_df.merge(other_comps, on="condition_concept_id", how="left")

# Save to CSV
merged_df.to_csv("None_Group_Other_Complications.csv", index=False)

# Step 5: Show summary
comp_summary = merged_df["Complication"].value_counts()
print("🩺 Other complications among 'None' participants:")
print(comp_summary.head(15))


In [ ]:
# Count number of participants per complication
comp_summary = other_comps["Complication"].value_counts().reset_index()
comp_summary.columns = ["Complication", "Number of Participants"]
comp_summary.to_csv("All_476_Other_Complications.csv", index=False)

print("✅ Saved summary for 476 participants' other complications.")


---
## Step 7 — Final MWU Tests & Results Export

**What this does:**  
Runs the complete Mann-Whitney U test battery across all lab–complication
combinations and exports results for visualisation.

**Output:** `out_complications/complication_lab_mwu_results.csv`

In [ ]:
import pandas as pd

df = pd.read_csv("All_476_Other_Complications.csv")
print("📄 Column names in the file:")
print(df.columns.tolist())



In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from IPython.display import display

# Step 1: Load data
labs = pd.read_csv("lab_measurements_wide_with_units.csv")
complications = pd.read_csv("SCD_Complication_Severity_Only.csv")

# Step 2: Merge on person_id
merged = pd.merge(complications, labs, on="person_id", how="inner")

# Step 3: Lab columns
lab_columns = labs.columns.drop("person_id")

# Step 4: Complication columns (binary only)
complication_columns = [col for col in complications.columns if col not in ["person_id", "severity_score"]]

# Step 5: Store results
results = []

for comp in complication_columns:
    unique_vals = merged[comp].dropna().unique()
    if set(unique_vals) != {0, 1}:
        continue  # Only binary complications

    n_total = merged.shape[0]
    n_yes = merged[merged[comp] == 1].shape[0]
    n_no = merged[merged[comp] == 0].shape[0]
    pct_yes = (n_yes / n_total) * 100

    for lab in lab_columns:
        group_yes = merged[merged[comp] == 1][lab].dropna()
        group_no = merged[merged[comp] == 0][lab].dropna()

        if len(group_yes) < 5 or len(group_no) < 5:
            continue

        try:
            stat, pval = mannwhitneyu(group_yes, group_no, alternative='two-sided')
        except Exception:
            pval = np.nan

        # Combine median and IQR
        median_yes = group_yes.median()
        q1_yes = group_yes.quantile(0.25)
        q3_yes = group_yes.quantile(0.75)
        iqr_yes = f"{round(median_yes,2)} ({round(q1_yes,2)}–{round(q3_yes,2)})"

        median_no = group_no.median()
        q1_no = group_no.quantile(0.25)
        q3_no = group_no.quantile(0.75)
        iqr_no = f"{round(median_no,2)} ({round(q1_no,2)}–{round(q3_no,2)})"

        severity_yes = merged.loc[merged[comp] == 1, "severity_score"].median()
        severity_no = merged.loc[merged[comp] == 0, "severity_score"].median()

        results.append({
            "Complication": comp,
            "Lab Test": lab,
            "n (Yes)": len(group_yes),
            "n (No)": len(group_no),
            "% with Complication": round(pct_yes, 1),
            "Lab (Yes: Median [IQR])": iqr_yes,
            "Lab (No: Median [IQR])": iqr_no,
            "Severity (Yes)": severity_yes,
            "Severity (No)": severity_no,
            "p-value": round(pval, 4)
        })

# Step 6: Create DataFrame
result_df = pd.DataFrame(results)
result_df = result_df.sort_values(by="p-value")

# Step 7: Display in notebook
display(result_df.head(10))

# Step 8: Save to CSV
result_df.to_csv("complication_lab_analysis_formatted.csv", index=False)

print("✅ Analysis complete with Median (IQR).")


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import os

# Step 1: Load data
labs = pd.read_csv("lab_measurements_wide_with_units.csv")
complications = pd.read_csv("SCD_Complication_Severity_Only.csv")

# Step 2: Merge
merged = pd.merge(complications, labs, on="person_id", how="inner")

# Step 3: Columns
lab_columns = labs.columns.drop("person_id")
complication_columns = [col for col in complications.columns if col not in ["person_id", "severity_score"]]

# Step 4: Output folder
os.makedirs("complication_reports", exist_ok=True)

# Step 5: Loop through complications
for comp in complication_columns:
    unique_vals = merged[comp].dropna().unique()
    if set(unique_vals) != {0, 1}:
        continue

    group_yes = merged[merged[comp] == 1]
    group_no = merged[merged[comp] == 0]

    if group_yes.shape[0] < 5 or group_no.shape[0] < 5:
        continue

    records = []

    for lab in lab_columns:
        values_yes = group_yes[lab].dropna()
        values_no = group_no[lab].dropna()

        if len(values_yes) < 5 or len(values_no) < 5:
            continue

        try:
            stat, pval = mannwhitneyu(values_yes, values_no, alternative='two-sided')
        except Exception:
            continue

        median_yes = round(values_yes.median(), 2)
        median_no = round(values_no.median(), 2)
        n_yes = len(values_yes)
        n_no = len(values_no)

        records.append({
            "Lab Test": lab,
            "Median (Yes)": median_yes,
            "Median (No)": median_no,
            "p-value": round(pval, 4),
            "n (Yes)": n_yes,
            "n (No)": n_no
        })

    if not records:
        continue

    summary_df = pd.DataFrame(records)
    summary_df = summary_df.sort_values(by="p-value")

    # Clean filename
    clean_comp = comp.replace("/", "_").replace(" ", "_").replace(":", "").replace(".", "").replace(",", "")

    # Save CSV
    csv_path = f"complication_reports/{clean_comp}.csv"
    summary_df.to_csv(csv_path, index=False)

    # Save HTML (for easy reading in browser or report)
    html_path = f"complication_reports/{clean_comp}.html"
    summary_df.to_html(html_path, index=False)

    print(f"✅ Saved for {comp}: {csv_path} and {html_path}")

print("🎉 All complication reports generated.")


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import os

# Step 1: Load data
labs = pd.read_csv("lab_measurements_wide_with_units.csv")
complications = pd.read_csv("SCD_Complication_Severity_Only.csv")

# Step 2: Merge
merged = pd.merge(complications, labs, on="person_id", how="inner")

# Step 3: Columns
lab_columns = labs.columns.drop("person_id")
complication_columns = [col for col in complications.columns if col not in ["person_id", "severity_score"]]

# Step 4: Output folder
os.makedirs("complication_reports_with_IQR", exist_ok=True)

# Step 5: Loop through complications
for comp in complication_columns:
    unique_vals = merged[comp].dropna().unique()
    if set(unique_vals) != {0, 1}:
        continue

    group_yes = merged[merged[comp] == 1]
    group_no = merged[merged[comp] == 0]

    if group_yes.shape[0] < 5 or group_no.shape[0] < 5:
        continue

    records = []

    for lab in lab_columns:
        values_yes = group_yes[lab].dropna()
        values_no = group_no[lab].dropna()

        if len(values_yes) < 5 or len(values_no) < 5:
            continue

        try:
            stat, pval = mannwhitneyu(values_yes, values_no, alternative='two-sided')
        except Exception:
            continue

        # Median and IQR
        med_yes = round(values_yes.median(), 2)
        q1_yes = round(values_yes.quantile(0.25), 2)
        q3_yes = round(values_yes.quantile(0.75), 2)
        iqr_yes = f"{med_yes} ({q1_yes}–{q3_yes})"

        med_no = round(values_no.median(), 2)
        q1_no = round(values_no.quantile(0.25), 2)
        q3_no = round(values_no.quantile(0.75), 2)
        iqr_no = f"{med_no} ({q1_no}–{q3_no})"

        n_yes = len(values_yes)
        n_no = len(values_no)

        records.append({
            "Lab Test": lab,
            "Median (IQR) – Yes": iqr_yes,
            "Median (IQR) – No": iqr_no,
            "p-value": round(pval, 4),
            "n (Yes)": n_yes,
            "n (No)": n_no
        })

    if not records:
        continue

    summary_df = pd.DataFrame(records)
    summary_df = summary_df.sort_values(by="p-value")

    # Clean filename
    clean_comp = comp.replace("/", "_").replace(" ", "_").replace(":", "").replace(".", "").replace(",", "")

    # Save CSV
    csv_path = f"complication_reports_with_IQR/{clean_comp}.csv"
    summary_df.to_csv(csv_path, index=False)

    # Save HTML
    html_path = f"complication_reports_with_IQR/{clean_comp}.html"
    summary_df.to_html(html_path, index=False)

    print(f"✅ Saved: {csv_path} and {html_path}")

print("🎉 All complication reports with Median (IQR) generated.")


---
## Step 8 — KMeans Clustering for Patient Stratification

**What this does:**  
Groups the 476 SCD participants into 3 clusters (k=3) based on their
lab measurement profiles using KMeans clustering.

**Pipeline:**
1. Impute missing values (median imputation)
2. Standardise all lab values (StandardScaler)
3. Apply KMeans (k=3)
4. Reduce to 2D with PCA for visualisation

**Expected output:** Cluster assignments per participant + PCA scatter plot.

In [ ]:
import pandas as pd
import os

# Load datasets
labs = pd.read_csv("lab_measurements_wide_with_units.csv")
complications = pd.read_csv("SCD_Complication_Severity_Only.csv")

# Merge on person_id
merged = pd.merge(complications, labs, on="person_id", how="inner")

# Extract lab and complication columns
lab_columns = labs.columns.drop("person_id")
complication_columns = [col for col in complications.columns if col not in ["person_id", "severity_score"]]

# Create output folder
os.makedirs("lab_availability_by_complication", exist_ok=True)

# Loop over complications
for comp in complication_columns:
    unique_vals = merged[comp].dropna().unique()
    if set(unique_vals) != {0, 1}:
        continue

    df_yes = merged[merged[comp] == 1]
    df_no = merged[merged[comp] == 0]

    counts = []

    for lab in lab_columns:
        n_yes = df_yes[lab].notna().sum()
        n_no = df_no[lab].notna().sum()
        counts.append({
            "Lab Test": lab,
            "Available in Yes group": n_yes,
            "Available in No group": n_no,
        })

    count_df = pd.DataFrame(counts)
    count_df = count_df.sort_values(by="Available in Yes group", ascending=False)

    # Clean filename
    clean_name = comp.replace("/", "_").replace(" ", "_").replace(":", "").replace(".", "").replace(",", "")
    
    # Save CSV and HTML
    count_df.to_csv(f"lab_availability_by_complication/{clean_name}_availability.csv", index=False)
    count_df.to_html(f"lab_availability_by_complication/{clean_name}_availability.html", index=False)

    print(f"✅ Saved availability summary for: {comp}")

print("📊 Done! All lab availability summaries are ready.")


In [ ]:
print(merged.columns.tolist())


In [ ]:
# Summarise clinical/lab variables per cluster# Summary statistics by cluster
cluster_summary = df_cluster.groupby('Cluster').mean().round(2)
cluster_summary

df_cluster.groupby('Cluster').mean()


---
## Step 9 — Build ML-Ready Feature Matrix

**What this does:**  
Merges demographics, lab values, complications, and severity scores
into a single ML-ready feature matrix (`ml_ready_latest.csv`).

This file can be used directly for supervised or unsupervised machine learning.

In [ ]:
# Summary statistics by cluster
cluster_summary = df_cluster.groupby('Cluster').mean().round(2)
cluster_summary


In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np

# Step 1: Load the data files
demo = pd.read_csv("SCD_Demographics.csv")
labs = pd.read_csv("lab_measurements_wide_with_units.csv")
comps = pd.read_csv("SCD_Top10_Complication_Details.csv")

# Step 2: Merge on person_id
df = demo.merge(labs, on="person_id", how="inner")
df = df.merge(comps, on="person_id", how="inner")

# Step 3: Calculate age
df['date_of_birth'] = pd.to_datetime(df['date_of_birth']).dt.tz_localize(None)
ref_date = pd.to_datetime("2025-07-01")
df['age'] = (ref_date - df['date_of_birth']).dt.days / 365.25

# Step 4: Rename key variables
df.rename(columns={
    'sex_at_birth': 'sex',
    'Systolic blood pressure': 'systolic_bp',
    'Diastolic blood pressure': 'diastolic_bp',
    'Heart rate': 'heart_rate',
    'Hemoglobin [Mass/volume] in Blood': 'hemoglobin',
    'Hemoglobin F/Hemoglobin.total in Blood': 'hbf_percent',
    'Leukocytes [#/volume] in Blood by Automated count': 'wbc',
    'Neutrophils/100 leukocytes in Blood by Automated count': 'neutrophils',
    'Monocytes/100 leukocytes in Blood by Automated count': 'monocytes',
    'Basophils/100 leukocytes in Blood by Automated count': 'basophils',
    'top10_complication_count': 'complication_count'
}, inplace=True)

# Step 5: Add precomputed cluster labels
# 🔁 Replace this random assignment with your actual cluster assignments
df['Cluster'] = np.random.choice([0, 1], size=len(df))  # Replace with real KMeans output if available

# Step 6: Manually map cluster to phenotype label (you can swap 0/1 depending on your model)
phenotype_map = {
    0: "Severe phenotype",
    1: "Mild phenotype"
}
df['Phenotype_Label'] = df['Cluster'].map(phenotype_map)

# Step 7: List of complication variables
complication_cols = [
    'dactylitis', 'restrictive_lung_disease', 'preeclampsia', 'sickle_retinopathy',
    'osteomyelitis', 'aplastic_crisis', 'liver_cirrhosis', 'vitreous_haemorrhage',
    'hand_foot_syndrome', 'pulmonary_hypertension'
]

# Step 8: Select variables for summary
summary_vars = [
    'age', 'sex', 'systolic_bp', 'diastolic_bp', 'heart_rate',
    'hemoglobin', 'hbf_percent', 'wbc', 'neutrophils',
    'monocytes', 'basophils', 'complication_count'
] + complication_cols

# Step 9: Convert sex to numeric
df['sex'] = df['sex'].map({'Male': 0, 'Female': 1})

# Step 10: Group and summarize
cluster_summary = df.groupby('Phenotype_Label')[summary_vars].mean().round(2)

# Step 11: Save and print
cluster_summary.to_csv("Phenotype_Cluster_Summary.csv")
print(cluster_summary)


---
## Step 10 — Extended Lab Extraction (Standardised Concept IDs)

**What this does:**  
Re-extracts lab measurements using a standardised set of concept IDs
(one ID per analyte) to ensure consistency across different lab naming conventions
in the All of Us dataset.

**Outputs to `./out/` folder:**
- `latest_long.csv` — long format with latest value per person per analyte
- `ml_ready_latest.csv` — wide format ML feature matrix
- `summary_medians.csv` — cohort-level median per analyte
- Matplotlib charts for each analyte distribution

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Load the uploaded files
demo = pd.read_csv("SCD_Demographics.csv")
labs = pd.read_csv("lab_measurements_wide_with_units.csv")
comps = pd.read_csv("SCD_Top10_Complication_Details.csv")

# Step 2: Merge all on 'person_id'
df = demo.merge(labs, on='person_id', how='inner')
df = df.merge(comps, on='person_id', how='inner')

# Step 3: Compute age
df['date_of_birth'] = pd.to_datetime(df['date_of_birth']).dt.tz_localize(None)
reference_date = pd.to_datetime('2025-07-01')
df['age'] = (reference_date - df['date_of_birth']).dt.days / 365.25

# Step 4: Rename columns
df.rename(columns={
    'sex_at_birth': 'sex',
    'Systolic blood pressure': 'systolic_bp',
    'Diastolic blood pressure': 'diastolic_bp',
    'Heart rate': 'heart_rate',
    'Hemoglobin [Mass/volume] in Blood': 'hemoglobin',
    'Hemoglobin F/Hemoglobin.total in Blood': 'hbf_percent',
    'Leukocytes [#/volume] in Blood by Automated count': 'wbc',
    'Neutrophils/100 leukocytes in Blood by Automated count': 'neutrophils',
    'Monocytes/100 leukocytes in Blood by Automated count': 'monocytes',
    'Basophils/100 leukocytes in Blood by Automated count': 'basophils',
    'top10_complication_count': 'complication_count'
}, inplace=True)

# Step 5: Select features for clustering including individual complications
complication_cols = [
    'dactylitis', 'restrictive_lung_disease', 'preeclampsia',
    'sickle_retinopathy', 'osteomyelitis', 'aplastic_crisis',
    'liver_cirrhosis', 'vitreous_haemorrhage', 'hand_foot_syndrome',
    'pulmonary_hypertension'
]

features = [
    'age', 'sex', 'systolic_bp', 'diastolic_bp', 'heart_rate',
    'hemoglobin', 'hbf_percent', 'wbc', 'neutrophils',
    'monocytes', 'basophils', 'complication_count'
] + complication_cols

# Encode sex
df['sex'] = df['sex'].map({'Male': 0, 'Female': 1})

# Step 6: Impute and scale
df_cluster = df[features].copy()
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(df_cluster)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Step 7: KMeans clustering
kmeans = KMeans(n_clusters=2, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Optional: label clusters based on average severity
cluster_means = df.groupby('Cluster')['complication_count'].mean()
severity_labels = ['Mild phenotype', 'Severe phenotype'] if cluster_means[0] < cluster_means[1] else ['Severe phenotype', 'Mild phenotype']
df['Phenotype_Label'] = df['Cluster'].map({0: severity_labels[0], 1: severity_labels[1]})

# Step 8: PCA for visualization
pca = PCA(n_components=2)
pca_vals = pca.fit_transform(X_scaled)
df['PC1'] = pca_vals[:, 0]
df['PC2'] = pca_vals[:, 1]

# Step 9: Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='PC1', y='PC2', hue='Phenotype_Label', palette='Set1')
plt.title("Unsupervised Clustering of SCD Patients with Complications (PCA)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

# Step 10: Cluster summary
cluster_summary = df.groupby('Phenotype_Label')[features].mean().round(2)

print(cluster_summary)
# OR save it to a CSV if you prefer:
cluster_summary.to_csv("cluster_summary_with_complications.csv", index=True)




In [ ]:
import os
import pandas as pd

demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df['person_id'].astype(str).unique().tolist()
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])
cdr = os.environ["WORKSPACE_CDR"]

targets = {
    "Hemoglobin": ["%hemoglobin%", "% hgb%"],
    "Fetal hemoglobin": ["%fetal hemoglobin%", "% hbf %", "hbf%"],
    "Lactate dehydrogenase": ["%lactate dehydrogenase%", "%ldh%"],
    "Bilirubin, total": ["%bilirubin total%", "%total bilirubin%", "%bilirubin, total%"],
    "Bilirubin, direct": ["%bilirubin direct%", "%direct bilirubin%", "%bilirubin, direct%"],
    "Creatinine": ["%creatinine%"],
    "Reticulocyte count": ["%reticulocyte%"],
    "Hematocrit": ["%hematocrit%", "%packed cell volume%"],
    "Mean corpuscular volume": ["%mean corpuscular volume%", "% mcv %", "mcv%"],
    "Mean corpuscular haemoglobin concentration": [
        "%mean corpuscular hemoglobin concentration%", "% mchc %", "mchc%"
    ],
    "Lymphocytes": ["%lymphocyte%"],
    "Basophils": ["%basophil%"],
    "Eosinophils": ["%eosinophil%"],
    "Monocytes": ["%monocyte%"],
    "Neutrophils": ["%neutrophil%"],
    "Platelets": ["%platelet%"],
    "Systolic blood pressure": ["%systolic blood pressure%"],
    "Diastolic blood pressure": ["%diastolic blood pressure%"],
    "Heart rate": ["%heart rate%"],
}

def like_clause(col, pats):
    return " OR ".join([f"LOWER({col}) LIKE '{p.lower()}'" for p in pats])

def resolver_sql(pats):
    return f"""
WITH cohort AS (
  SELECT person_id FROM UNNEST([{formatted_ids}]) AS person_id
),
candidates AS (
  SELECT concept_id, concept_name, standard_concept
  FROM `{cdr}.concept`
  WHERE domain_id = 'Measurement'
    AND ({like_clause('concept_name', pats)})
),
std AS (
  SELECT m.measurement_concept_id AS concept_id,
         c.concept_name, COUNT(*) AS n_rows
  FROM `{cdr}.measurement` m
  JOIN cohort co ON co.person_id = CAST(m.person_id AS STRING)
  JOIN candidates c ON c.concept_id = m.measurement_concept_id
  WHERE c.standard_concept = 'S'
  GROUP BY 1,2
),
src_used AS (
  SELECT m.measurement_source_concept_id AS source_concept_id,
         c.concept_name AS source_concept_name, COUNT(*) AS n_rows
  FROM `{cdr}.measurement` m
  JOIN cohort co ON co.person_id = CAST(m.person_id AS STRING)
  JOIN candidates c ON c.concept_id = m.measurement_source_concept_id
  WHERE m.measurement_source_concept_id IS NOT NULL
  GROUP BY 1,2
),
src_to_std AS (
  SELECT s.source_concept_id, s.source_concept_name, s.n_rows,
         cr.concept_id_2 AS standard_concept_id
  FROM src_used s
  LEFT JOIN `{cdr}.concept_relationship` cr
    ON cr.concept_id_1 = s.source_concept_id
   AND cr.relationship_id = 'Maps to'
),
unioned AS (
  SELECT concept_id AS standard_concept_id, concept_name, n_rows FROM std
  UNION ALL
  SELECT standard_concept_id, c2.concept_name, n_rows
  FROM src_to_std u
  LEFT JOIN `{cdr}.concept` c2 ON c2.concept_id = u.standard_concept_id
)
SELECT standard_concept_id AS concept_id,
       ANY_VALUE(concept_name) AS concept_name,
       SUM(n_rows) AS total_rows
FROM unioned
WHERE standard_concept_id IS NOT NULL
GROUP BY standard_concept_id
ORDER BY total_rows DESC, concept_id
"""

# build lab_map: top 1–3 concepts per analyte
TOP_N = 3
lab_map = {}
for label, pats in targets.items():
    df = pd.read_gbq(resolver_sql(pats), dialect="standard",
                     use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))
    if df.empty:
        continue
    for cid in df.head(TOP_N)["concept_id"].tolist():
        lab_map[int(cid)] = label

print("✅ Final lab_map (cohort-specific IDs):")
for cid, lbl in lab_map.items():
    print(f"{cid}: {lbl}")


In [ ]:
import os
import pandas as pd

# --- 1) Load your 476-person cohort ---
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df['person_id'].astype(str).unique().tolist()
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])
cdr = os.environ["WORKSPACE_CDR"]

# --- 2) Cohort-specific lab_map with Fetal hemoglobin included ---
lab_map = {
    # Hemoglobin
    3000963: "Hemoglobin",
    40758903: "Hemoglobin",
    3011397: "Hemoglobin",

    # Fetal hemoglobin
    3018738: "Fetal hemoglobin",   # OMOP standard for HbF — if your concept search shows alternates, add them here
    # If your concept search found others, e.g.:
    # 40762722: "Fetal hemoglobin",
    # 40786612: "Fetal hemoglobin",

    # LDH
    3022250: "Lactate dehydrogenase",
    3016436: "Lactate dehydrogenase",
    3034022: "Lactate dehydrogenase",

    # Bilirubin
    3661905: "Bilirubin, total",
    4230543: "Bilirubin, total",
    2212099: "Bilirubin, direct",
    3045524: "Bilirubin, direct",
    4216632: "Bilirubin, direct",

    # Creatinine
    3016723: "Creatinine",
    3018311: "Creatinine",
    46236952: "Creatinine",

    # Reticulocytes
    3023520: "Reticulocyte count",
    3007124: "Reticulocyte count",
    40763528: "Reticulocyte count",

    # Hematocrit
    3023314: "Hematocrit",
    3009542: "Hematocrit",
    3034976: "Hematocrit",

    # MCV
    3023599: "Mean corpuscular volume",
    3024731: "Mean corpuscular volume",
    37393851: "Mean corpuscular volume",

    # MCHC
    3009744: "Mean corpuscular haemoglobin concentration",
    3003338: "Mean corpuscular haemoglobin concentration",
    3051314: "Mean corpuscular haemoglobin concentration",

    # Differential
    3004327: "Lymphocytes",
    3037511: "Lymphocytes",
    3019198: "Lymphocytes",

    3013869: "Basophils",
    3013429: "Basophils",
    3006315: "Basophils",

    3028615: "Eosinophils",
    3010457: "Eosinophils",
    3006504: "Eosinophils",

    3033575: "Monocytes",
    3011948: "Monocytes",
    3001604: "Monocytes",

    3013650: "Neutrophils",
    3008342: "Neutrophils",
    3017732: "Neutrophils",

    # Platelets
    3024929: "Platelets",
    3043111: "Platelets",
    3022217: "Platelets",

    # Blood pressure
    3004249: "Systolic blood pressure",
    4152194: "Systolic blood pressure",
    3028737: "Systolic blood pressure",

    3012888: "Diastolic blood pressure",
    4154790: "Diastolic blood pressure",
    21490851: "Diastolic blood pressure",

    # Heart rate
    3027018: "Heart rate",
    3001376: "Heart rate",
    4239408: "Heart rate"
}

concept_ids_str = ", ".join(map(str, lab_map.keys()))

# --- 3) Query all measurements for these people & concepts ---
lab_df = pd.read_gbq(
    f"""
    SELECT person_id, measurement_concept_id, measurement_date, value_as_number
    FROM `{cdr}.measurement`
    WHERE CAST(person_id AS STRING) IN ({formatted_ids})
      AND measurement_concept_id IN ({concept_ids_str})
    """,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)

# --- 4) Map to analyte labels ---
lab_df["lab_test"] = lab_df["measurement_concept_id"].map(lab_map)

# --- 5) Get latest value per person/analyte ---
lab_df = lab_df.sort_values(["person_id", "lab_test", "measurement_date"])
latest = lab_df.groupby(["person_id", "lab_test"], as_index=False).tail(1)

# --- 6) Pivot to wide format ---
ml_ready = latest.pivot(index="person_id", columns="lab_test", values="value_as_number").reset_index()
ml_ready.columns.name = None  # remove top label

# --- 7) Ensure all analyte columns exist ---
desired_cols = ["person_id"] + list(dict.fromkeys(lab_map.values()))
for col in desired_cols:
    if col not in ml_ready.columns:
        ml_ready[col] = pd.NA
ml_ready = ml_ready[desired_cols]

print("✅ Final ML-ready shape:", ml_ready.shape)
print(ml_ready.head(10))


In [ ]:
import pandas as pd
import os

# --- Load your cohort ---
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df["person_id"].astype(str).unique().tolist()
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])
cdr = os.environ["WORKSPACE_CDR"]

# --- Your cohort-specific lab_map (multiple IDs per analyte) ---
lab_map = {
    3000963: "Hemoglobin",
    40758903: "Hemoglobin",
    3011397: "Hemoglobin",
    3018738: "Fetal hemoglobin",
    3022250: "Lactate dehydrogenase",
    3016436: "Lactate dehydrogenase",
    3034022: "Lactate dehydrogenase",
    3661905: "Bilirubin, total",
    4230543: "Bilirubin, total",
    2212099: "Bilirubin, direct",
    3045524: "Bilirubin, direct",
    4216632: "Bilirubin, direct",
    3016723: "Creatinine",
    3018311: "Creatinine",
    46236952: "Creatinine",
    3023520: "Reticulocyte count",
    3007124: "Reticulocyte count",
    40763528: "Reticulocyte count",
    3023314: "Hematocrit",
    3009542: "Hematocrit",
    3034976: "Hematocrit",
    3023599: "Mean corpuscular volume",
    3024731: "Mean corpuscular volume",
    37393851: "Mean corpuscular volume",
    3009744: "Mean corpuscular haemoglobin concentration",
    3003338: "Mean corpuscular haemoglobin concentration",
    3051314: "Mean corpuscular haemoglobin concentration",
    3004327: "Lymphocytes",
    3037511: "Lymphocytes",
    3019198: "Lymphocytes",
    3013869: "Basophils",
    3013429: "Basophils",
    3006315: "Basophils",
    3028615: "Eosinophils",
    3010457: "Eosinophils",
    3006504: "Eosinophils",
    3033575: "Monocytes",
    3011948: "Monocytes",
    3001604: "Monocytes",
    3013650: "Neutrophils",
    3008342: "Neutrophils",
    3017732: "Neutrophils",
    3024929: "Platelets",
    3043111: "Platelets",
    3022217: "Platelets",
    3004249: "Systolic blood pressure",
    4152194: "Systolic blood pressure",
    3028737: "Systolic blood pressure",
    3012888: "Diastolic blood pressure",
    4154790: "Diastolic blood pressure",
    21490851: "Diastolic blood pressure",
    3027018: "Heart rate",
    3001376: "Heart rate",
    4239408: "Heart rate",
}

# ---------------------------
# Step 1: Count usage of each concept_id in cohort
# ---------------------------
concept_counts = []

for cid, analyte in lab_map.items():
    sql = f"""
    SELECT COUNT(DISTINCT person_id) AS n_patients
    FROM `{cdr}.measurement`
    WHERE CAST(person_id AS STRING) IN ({formatted_ids})
      AND measurement_concept_id = {cid}
    """
    count_df = pd.read_gbq(sql, dialect="standard",
                           use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))
    n_patients = int(count_df.iloc[0, 0])
    concept_counts.append((analyte, cid, n_patients))

concept_count_df = pd.DataFrame(concept_counts, columns=["analyte", "concept_id", "n_patients"])

# ---------------------------
# Step 2: Pick the most common concept_id for each analyte
# ---------------------------
most_common_ids = (
    concept_count_df.sort_values(["analyte", "n_patients"], ascending=[True, False])
    .groupby("analyte", as_index=False)
    .first()
)

# ---------------------------
# Step 3: For each analyte, get most recent measurement for that chosen ID per person
# and calculate average
# ---------------------------
results = []

for _, row in most_common_ids.iterrows():
    analyte = row["analyte"]
    cid = row["concept_id"]

    sql = f"""
    WITH latest AS (
      SELECT person_id, value_as_number,
             ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
      FROM `{cdr}.measurement`
      WHERE CAST(person_id AS STRING) IN ({formatted_ids})
        AND measurement_concept_id = {cid}
    )
    SELECT person_id, value_as_number
    FROM latest
    WHERE rn = 1
    """
    meas_df = pd.read_gbq(sql, dialect="standard",
                          use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))
    
    avg_val = meas_df["value_as_number"].mean()
    n_available = meas_df["value_as_number"].notna().sum()
    results.append((analyte, cid, n_available, avg_val))

# ---------------------------
# Step 4: Final summary table
# ---------------------------
final_df = pd.DataFrame(results, columns=["Analyte", "Chosen concept_id", "Patients with value", "Average value"])
final_df = final_df.sort_values("Analyte").reset_index(drop=True)

print("✅ Summary of most common concept_id per analyte with average values:")
print(final_df)

# Optional: Save to CSV
# final_df.to_csv("lab_summary_most_common_concept.csv", index=False)


In [ ]:
import os
import pandas as pd

# ---------------------------
# 0) Load cohort (476 people)
# ---------------------------
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df["person_id"].astype(str).unique().tolist()
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])
cdr = os.environ["WORKSPACE_CDR"]

# ---------------------------
# 1) Your cohort-specific lab_map (MANY ids -> ONE analyte label)
#    Includes Fetal hemoglobin
# ---------------------------
lab_map_multi = {
    3000963: "Hemoglobin",
    40758903: "Hemoglobin",
    3011397: "Hemoglobin",

    3018738: "Fetal hemoglobin",   # add more HbF IDs here if you discover them

    3022250: "Lactate dehydrogenase",
    3016436: "Lactate dehydrogenase",
    3034022: "Lactate dehydrogenase",

    3661905: "Bilirubin, total",
    4230543: "Bilirubin, total",
    2212099: "Bilirubin, direct",
    3045524: "Bilirubin, direct",
    4216632: "Bilirubin, direct",

    3016723: "Creatinine",
    3018311: "Creatinine",
    46236952: "Creatinine",

    3023520: "Reticulocyte count",
    3007124: "Reticulocyte count",
    40763528: "Reticulocyte count",

    3023314: "Hematocrit",
    3009542: "Hematocrit",
    3034976: "Hematocrit",

    3023599: "Mean corpuscular volume",
    3024731: "Mean corpuscular volume",
    37393851: "Mean corpuscular volume",

    3009744: "Mean corpuscular haemoglobin concentration",
    3003338: "Mean corpuscular haemoglobin concentration",
    3051314: "Mean corpuscular haemoglobin concentration",

    3004327: "Lymphocytes",
    3037511: "Lymphocytes",
    3019198: "Lymphocytes",

    3013869: "Basophils",
    3013429: "Basophils",
    3006315: "Basophils",

    3028615: "Eosinophils",
    3010457: "Eosinophils",
    3006504: "Eosinophils",

    3033575: "Monocytes",
    3011948: "Monocytes",
    3001604: "Monocytes",

    3013650: "Neutrophils",
    3008342: "Neutrophils",
    3017732: "Neutrophils",

    3024929: "Platelets",
    3043111: "Platelets",
    3022217: "Platelets",

    3004249: "Systolic blood pressure",
    4152194: "Systolic blood pressure",
    3028737: "Systolic blood pressure",

    3012888: "Diastolic blood pressure",
    4154790: "Diastolic blood pressure",
    21490851: "Diastolic blood pressure",

    3027018: "Heart rate",
    3001376: "Heart rate",
    4239408: "Heart rate",
}

# Keep a stable analyte order
analyte_order = list(dict.fromkeys(lab_map_multi.values()))

# ---------------------------
# 2) Pick ONE concept_id per analyte (most patients using it)
# ---------------------------
# Count distinct patients per concept_id
counts = []
for cid, analyte in lab_map_multi.items():
    sql = f"""
    SELECT COUNT(DISTINCT person_id) AS n_patients
    FROM `{cdr}.measurement`
    WHERE CAST(person_id AS STRING) IN ({formatted_ids})
      AND measurement_concept_id = {cid}
    """
    df = pd.read_gbq(sql, dialect="standard",
                     use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))
    n = int(df.iloc[0, 0])
    counts.append((analyte, cid, n))

counts_df = pd.DataFrame(counts, columns=["analyte", "concept_id", "n_patients"])

# Choose the most-used concept_id per analyte
chosen = (
    counts_df.sort_values(["analyte", "n_patients"], ascending=[True, False])
             .groupby("analyte", as_index=False)
             .first()
)

# Build a ONE-to-ONE lab_map (analyte -> single concept id)
analyte_to_cid = dict(zip(chosen["analyte"], chosen["concept_id"]))
# And the inverse map for fast labeling
lab_map_single = {cid: analyte for analyte, cid in analyte_to_cid.items()}

print("📌 Chosen concept_id per analyte:")
print(pd.DataFrame.from_dict(analyte_to_cid, orient="index", columns=["concept_id"]).sort_index())

# ---------------------------
# 3) Pull only those chosen concept_ids for the cohort
# ---------------------------
concept_ids_str = ", ".join(str(x) for x in lab_map_single.keys())

sql = f"""
SELECT
  person_id,
  measurement_concept_id,
  measurement_date,
  value_as_number
FROM `{cdr}.measurement`
WHERE CAST(person_id AS STRING) IN ({formatted_ids})
  AND measurement_concept_id IN ({concept_ids_str})
"""
lab_df = pd.read_gbq(sql, dialect="standard",
                     use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))

# ---------------------------
# 4) Your exact steps 5/6/7/8
#    Step 5: Add readable names
# ---------------------------
lab_df["lab_test"] = lab_df["measurement_concept_id"].map(lab_map_single)

# Step 6: Keep only the latest measurement per patient per test
lab_df = lab_df.sort_values(by=["person_id", "lab_test", "measurement_date"])
latest_labs = lab_df.groupby(["person_id", "lab_test"], as_index=False).tail(1)

# Step 7: Pivot to wide format for ML
ml_ready = latest_labs.pivot(index="person_id", columns="lab_test", values="value_as_number").reset_index()
ml_ready.columns.name = None

# (Optional) ensure all analytes appear as columns, even if entirely missing
for a in analyte_order:
    if a not in ml_ready.columns and a in analyte_to_cid:
        ml_ready[a] = pd.NA
ml_ready = ml_ready[["person_id"] + [a for a in analyte_order if a in analyte_to_cid]]

# Step 8: Final preview
print("✅ Final ML-ready lab dataset shape:", ml_ready.shape)
print(ml_ready.head(10))

# ---------------------------
# 5) (Nice-to-have) Show a small summary table with averages
# ---------------------------
summary = []
for a, cid in analyte_to_cid.items():
    if a in ml_ready.columns:
        vals = ml_ready[a]
        summary.append({
            "Analyte": a,
            "Chosen concept_id": cid,
            "Patients with value": int(vals.notna().sum()),
            "Average value": float(vals.mean()) if vals.notna().any() else None
        })
summary_df = pd.DataFrame(summary).sort_values("Analyte").reset_index(drop=True)

print("\n📊 Summary (one concept_id per analyte; average of most recent values):")
print(summary_df)

# Optional: save outputs
# ml_ready.to_csv("ml_ready_latest_oneID_per_analyte.csv", index=False)
# summary_df.to_csv("summary_oneID_per_analyte.csv", index=False)


In [ ]:
# -*- coding: utf-8 -*-
"""
Standardize labs to one concept_id per analyte (most-used),
pull latest per person, compute medians, and export CSVs & charts.

Outputs (written to ./out/):
- ml_ready_latest.csv                : wide table (one row per person, latest value per analyte)
- latest_long.csv                    : long table (person_id, analyte, concept_id, date, value)
- summary_stats.csv                  : analyte-level stats (median, mean, n, min, max, std)
- chosen_concept_ids.csv             : analyte -> chosen concept_id (most-used)
- medians_bar.png                    : bar chart of analyte medians
- distributions/<analyte>_hist.png   : per-analyte histogram of latest values
- distributions/<analyte>_box.png    : per-analyte boxplot of latest values
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------------------
# 0) Setup
# ---------------------------
OUT_DIR = Path("out")
DIST_DIR = OUT_DIR / "distributions"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DIST_DIR.mkdir(parents=True, exist_ok=True)

cdr = os.environ["WORKSPACE_CDR"]  # e.g., 'aou-res-curation-R2024Q2R1'

# ---------------------------
# 1) Load cohort (476 people)
# ---------------------------
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df["person_id"].astype(str).unique().tolist()
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])

# ---------------------------
# 2) Your cohort-specific lab_map (MANY ids -> ONE analyte label)
#    Includes Fetal hemoglobin
# ---------------------------
lab_map_multi = {
    3000963: "Hemoglobin",
    40758903: "Hemoglobin",
    3011397: "Hemoglobin",

    3018738: "Fetal hemoglobin",  # add more HbF IDs here if your resolver found them

    3022250: "Lactate dehydrogenase",
    3016436: "Lactate dehydrogenase",
    3034022: "Lactate dehydrogenase",

    3661905: "Bilirubin, total",
    4230543: "Bilirubin, total",
    2212099: "Bilirubin, direct",
    3045524: "Bilirubin, direct",
    4216632: "Bilirubin, direct",

    3016723: "Creatinine",
    3018311: "Creatinine",
    46236952: "Creatinine",

    3023520: "Reticulocyte count",
    3007124: "Reticulocyte count",
    40763528: "Reticulocyte count",

    3023314: "Hematocrit",
    3009542: "Hematocrit",
    3034976: "Hematocrit",

    3023599: "Mean corpuscular volume",
    3024731: "Mean corpuscular volume",
    37393851: "Mean corpuscular volume",

    3009744: "Mean corpuscular haemoglobin concentration",
    3003338: "Mean corpuscular haemoglobin concentration",
    3051314: "Mean corpuscular haemoglobin concentration",

    3004327: "Lymphocytes",
    3037511: "Lymphocytes",
    3019198: "Lymphocytes",

    3013869: "Basophils",
    3013429: "Basophils",
    3006315: "Basophils",

    3028615: "Eosinophils",
    3010457: "Eosinophils",
    3006504: "Eosinophils",

    3033575: "Monocytes",
    3011948: "Monocytes",
    3001604: "Monocytes",

    3013650: "Neutrophils",
    3008342: "Neutrophils",
    3017732: "Neutrophils",

    3024929: "Platelets",
    3043111: "Platelets",
    3022217: "Platelets",

    3004249: "Systolic blood pressure",
    4152194: "Systolic blood pressure",
    3028737: "Systolic blood pressure",

    3012888: "Diastolic blood pressure",
    4154790: "Diastolic blood pressure",
    21490851: "Diastolic blood pressure",

    3027018: "Heart rate",
    3001376: "Heart rate",
    4239408: "Heart rate",
}

# Keep analyte order
analyte_order = list(dict.fromkeys(lab_map_multi.values()))

# ---------------------------
# 3) Choose ONE concept_id per analyte (most patients using it)
# ---------------------------
counts = []
for cid, analyte in lab_map_multi.items():
    sql = f"""
    SELECT COUNT(DISTINCT person_id) AS n_patients
    FROM `{cdr}.measurement`
    WHERE CAST(person_id AS STRING) IN ({formatted_ids})
      AND measurement_concept_id = {cid}
    """
    dfc = pd.read_gbq(sql, dialect="standard",
                      use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))
    n = int(dfc.iloc[0, 0])
    counts.append((analyte, cid, n))

counts_df = pd.DataFrame(counts, columns=["analyte", "concept_id", "n_patients"])
chosen = (counts_df.sort_values(["analyte", "n_patients"], ascending=[True, False])
                    .groupby("analyte", as_index=False)
                    .first())

# analyte -> single concept_id; and inverse
analyte_to_cid = dict(zip(chosen["analyte"], chosen["concept_id"]))
lab_map_single = {cid: analyte for analyte, cid in analyte_to_cid.items()}

# Save chosen concept IDs
pd.DataFrame.from_dict(analyte_to_cid, orient="index", columns=["concept_id"])\
  .rename_axis("Analyte").reset_index()\
  .sort_values("Analyte")\
  .to_csv(OUT_DIR / "chosen_concept_ids.csv", index=False)

# ---------------------------
# 4) Pull only the chosen concept_ids and keep LATEST per person/analyte
# ---------------------------
concept_ids_str = ", ".join(str(x) for x in lab_map_single.keys())
sql = f"""
SELECT
  person_id,
  measurement_concept_id,
  measurement_date,
  value_as_number
FROM `{cdr}.measurement`
WHERE CAST(person_id AS STRING) IN ({formatted_ids})
  AND measurement_concept_id IN ({concept_ids_str})
"""
lab_df = pd.read_gbq(sql, dialect="standard",
                     use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))

lab_df["lab_test"] = lab_df["measurement_concept_id"].map(lab_map_single)

# latest per person/analyte
lab_df = lab_df.sort_values(["person_id", "lab_test", "measurement_date"])
latest = lab_df.groupby(["person_id", "lab_test"], as_index=False).tail(1)

# Save the long latest rows
latest_long = latest[["person_id", "lab_test", "measurement_concept_id",
                      "measurement_date", "value_as_number"]].copy()
latest_long.rename(columns={
    "lab_test": "Analyte",
    "measurement_concept_id": "concept_id",
    "measurement_date": "date",
    "value_as_number": "value"
}, inplace=True)
latest_long.to_csv(OUT_DIR / "latest_long.csv", index=False)

# ---------------------------
# 5) Pivot to wide ML-ready table
# ---------------------------
ml_ready = latest.pivot(index="person_id", columns="lab_test", values="value_as_number").reset_index()
ml_ready.columns.name = None

# Ensure all analytes present as columns
for a in analyte_order:
    if a in analyte_to_cid and a not in ml_ready.columns:
        ml_ready[a] = pd.NA
ml_ready = ml_ready[["person_id"] + [a for a in analyte_order if a in analyte_to_cid]]
ml_ready.to_csv(OUT_DIR / "ml_ready_latest.csv", index=False)

print(f"✅ ML-ready saved -> {OUT_DIR / 'ml_ready_latest.csv'}")
print(f"✅ Latest long saved -> {OUT_DIR / 'latest_long.csv'}")
print(f"✅ Chosen IDs saved -> {OUT_DIR / 'chosen_concept_ids.csv'}")

# ---------------------------
# 6) Summary stats (median + extras)
# ---------------------------
summary_rows = []
for a, cid in analyte_to_cid.items():
    if a in ml_ready.columns:
        s = pd.to_numeric(ml_ready[a], errors="coerce").dropna()
        summary_rows.append({
            "Analyte": a,
            "concept_id": cid,
            "Patients with value": int(s.size),
            "Median": float(s.median()) if s.size else np.nan,
            "Mean": float(s.mean()) if s.size else np.nan,
            "Std": float(s.std(ddof=1)) if s.size > 1 else np.nan,
            "Min": float(s.min()) if s.size else np.nan,
            "Max": float(s.max()) if s.size else np.nan
        })

summary_df = pd.DataFrame(summary_rows).sort_values("Analyte").reset_index(drop=True)
summary_df.to_csv(OUT_DIR / "summary_stats.csv", index=False)
print(f"✅ Summary stats saved -> {OUT_DIR / 'summary_stats.csv'}")

# ---------------------------
# 7) Charts
#    IMPORTANT: per instructions, use matplotlib, one chart per file, no custom colors/styles
# ---------------------------

# 7a) Bar chart of medians
plt.figure()
plt.bar(summary_df["Analyte"], summary_df["Median"])
plt.title("Median of Latest Values per Analyte")
plt.xlabel("Analyte")
plt.ylabel("Median (units vary by analyte)")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "medians_bar.png")
plt.close()

# 7b) Per-analyte histograms and boxplots
for a in summary_df["Analyte"]:
    if a in ml_ready.columns:
        vals = pd.to_numeric(ml_ready[a], errors="coerce").dropna()
        if vals.empty:
            continue

        # Histogram
        plt.figure()
        plt.hist(vals, bins=30)
        plt.title(f"{a} - Histogram (Latest Values)")
        plt.xlabel(f"{a} (units vary)")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(DIST_DIR / f"{a.replace('/', '_')}_hist.png")
        plt.close()

        # Boxplot
        plt.figure()
        plt.boxplot(vals, vert=True)
        plt.title(f"{a} - Boxplot (Latest Values)")
        plt.ylabel(f"{a} (units vary)")
        plt.tight_layout()
        plt.savefig(DIST_DIR / f"{a.replace('/', '_')}_box.png")
        plt.close()

print(f" Charts saved -> {OUT_DIR / 'medians_bar.png'} and {DIST_DIR}/<analyte>_*.png")

# ---------------------------
# 8) Print quick peek
# ---------------------------
print("\n Summary (head):")
print(summary_df.head(10))
print("\n Chosen concept IDs:")
print(pd.DataFrame.from_dict(analyte_to_cid, orient="index", columns=["concept_id"]).sort_index())


In [ ]:
# -*- coding: utf-8 -*-
"""
Extract latest lab/vital measurements for a fixed set of concept IDs
(one ID per analyte as requested), and compute cohort averages.

Outputs (to ./out):
- latest_long.csv  : person_id, analyte, concept_id, measurement_date, value
- summary_stats.csv: analyte-level stats (n, mean, median)
- (optional) ml_ready_latest.csv: wide table (one row per person)
"""

import os
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------
# 0) Setup
# ---------------------------
OUT_DIR = Path("out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

cdr = os.environ["WORKSPACE_CDR"]  # e.g., 'aou-res-curation-R2024Q2R1'

# ---------------------------
# 1) Load cohort (476 people)
# ---------------------------
demographics_df = pd.read_csv("SCD_Demographics.csv")
person_ids = demographics_df["person_id"].astype(str).unique().tolist()
formatted_ids = ", ".join([f"'{pid}'" for pid in person_ids])

# ---------------------------
# 2) ONE concept_id per analyte
#    (exactly as you provided; others filled with standard single choices)
# ---------------------------
lab_map_single = {
    # --- Your provided IDs ---
    3000963: "Hemoglobin",
    3018738:  "Fetal hemoglobin",
    3020416: "Red blood cell count",
    3024929:  "Platelets",

    3000905:  "WBC (Leukocytes)",   # your WBC
    3037511:   "Lymphocytes",
    3013869:  "Basophils",
    3010451:  "Eosinophils",
    3011948:  "Monocytes",
    3008342:  "Neutrophils",

    3004249:  "Systolic blood pressure",
    3012888:  "Diastolic blood pressure",
    3027018:  "Heart rate",
    3016723: "Creatinine",

    # --- Single-ID picks for remaining SCD-relevant labs ---
    3022250:  "Lactate dehydrogenase",
    3661905:  "Bilirubin, total",
    2212099:  "Bilirubin, direct",
    3023520:  "Reticulocyte count",
    3023314:  "Hematocrit",
    3023599:  "Mean corpuscular volume",
    3009744:  "Mean corpuscular haemoglobin concentration",
}

# Keep a stable analyte order
analyte_order = list(dict.fromkeys(lab_map_single.values()))
concept_ids_str = ", ".join(str(cid) for cid in lab_map_single.keys())

# ---------------------------
# 3) Query measurement rows for these people & concept IDs
# ---------------------------
sql = f"""
SELECT
  person_id,
  measurement_concept_id,
  measurement_date,
  value_as_number
FROM `{cdr}.measurement`
WHERE CAST(person_id AS STRING) IN ({formatted_ids})
  AND measurement_concept_id IN ({concept_ids_str})
"""
lab_df = pd.read_gbq(
    sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
)

# Map concept_id -> analyte label
lab_df["analyte"] = lab_df["measurement_concept_id"].map(lab_map_single)

# ---------------------------
# 4) Keep ONLY the most recent per person/analyte
# ---------------------------
lab_df = lab_df.sort_values(["person_id", "analyte", "measurement_date"])
latest = lab_df.groupby(["person_id", "analyte"], as_index=False).tail(1)

# ---------------------------
# 5) Save LONG table with person_id, analyte, concept_id, date, value
# ---------------------------
latest_long = latest.rename(
    columns={
        "measurement_concept_id": "concept_id",
        "measurement_date": "measurement_date",
        "value_as_number": "value",
    }
)[["person_id", "analyte", "concept_id", "measurement_date", "value"]]

latest_long.to_csv(OUT_DIR / "latest_long.csv", index=False)
print(f"✅ Saved: {OUT_DIR / 'latest_long.csv'}  (rows: {len(latest_long)})")

# ---------------------------
# 6) Build summary (average/median per analyte)
# ---------------------------
summary = (
    latest_long
    .groupby("analyte", as_index=False)
    .agg(
        **{
            "concept_id": ("concept_id", "first"),   # single fixed ID per analyte
            "patients_with_value": ("value", lambda x: x.notna().sum()),
            "mean": ("value", "mean"),
            "median": ("value", "median"),
        }
    )
)

# Order & tidy
summary = summary.sort_values("analyte").reset_index(drop=True)
summary["mean"] = summary["mean"].astype(float).round(4)
summary["median"] = summary["median"].astype(float).round(4)

summary.to_csv(OUT_DIR / "summary_stats.csv", index=False)
print(f"Saved: {OUT_DIR / 'summary_stats.csv'}")

# Quick print
print("\n Summary (head):")
print(summary.head(12))

# ---------------------------
# 7) (Optional) Wide ML table (one row per person)
# ---------------------------
ml_ready = latest_long.pivot(index="person_id", columns="analyte", values="value").reset_index()
ml_ready.columns.name = None

# ensure all analytes appear, in order
for a in analyte_order:
    if a not in ml_ready.columns:
        ml_ready[a] = np.nan
ml_ready = ml_ready[["person_id"] + analyte_order]

ml_ready.to_csv(OUT_DIR / "ml_ready_latest.csv", index=False)
print(f" Saved: {OUT_DIR / 'ml_ready_latest.csv'}  (shape: {ml_ready.shape})") 




In [ ]:
import matplotlib.pyplot as plt

# ---------------------------
# Median bar chart
# ---------------------------
plt.figure(figsize=(12, 6))
plt.bar(summary["analyte"], summary["median"], color="skyblue", edgecolor="black")
plt.xticks(rotation=75, ha="right")
plt.ylabel("Median Value")
plt.title("Median of Most Recent Lab Measurements per Analyte (Cohort)")
plt.tight_layout()
plt.savefig(OUT_DIR / "median_bar.png", dpi=300)
plt.show()

print(f"📊 Saved median bar chart: {OUT_DIR / 'median_bar.png'}")


---
## Step 11 — Complication-Level MWU Results Visualisation

**What this does:**  
Loads the MWU results CSV and creates visualisations showing which
lab tests are significantly different across each complication:
- Dot plots showing p-values per complication
- Heatmap of significance across all lab–complication pairs
- Volcano-style plot highlighting significant findings

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# ---- 1) Load your new saved file ----
csv_path = "out/ml_ready_latest.csv"   # change if you saved elsewhere
df = pd.read_csv(csv_path)

# Keep only numeric lab columns (exclude person_id)
lab_tests = [c for c in df.columns if c != "person_id"]

# Coerce to numeric just in case some columns are objects
for c in lab_tests:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ---- 2) Set up subplot grid ----
n = len(lab_tests)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
axes = axes.flatten()

# ---- 3) Plot each lab ----
for i, lab in enumerate(lab_tests):
    ax = axes[i]
    values = df[lab].dropna().values

    if values.size < 10:
        ax.set_title(f"{lab} (Insufficient data: n={values.size})", fontsize=10)
        ax.axis("off")
        continue

    # Fit a normal (for visual reference only)
    mu, std = norm.fit(values)
    x = np.linspace(values.min(), values.max(), 200)
    p = norm.pdf(x, mu, std)

    # Histogram + normal curve + median line
    ax.hist(values, bins=20, density=True, alpha=0.6, edgecolor="black")
    ax.plot(x, p, "k", linewidth=1.2)
    ax.axvline(np.median(values), linestyle="--", linewidth=1.2)

    ax.set_title(f"{lab}\nμ={mu:.2f}, σ={std:.2f}, median={np.median(values):.2f}", fontsize=10)
    ax.set_ylabel("Density")
    ax.set_xlabel("Value")

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.suptitle("Lab Measurement Distributions (SCD Participants)", fontsize=16, y=1.02)
plt.subplots_adjust(top=0.92)

# ---- 4) Save combined figure ----
out_path = "out/combined_lab_distributions.png"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, dpi=300)
plt.show()

print(f" Saved: {out_path}")


In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import mannwhitneyu

# ========= Config: point to YOUR new files =========
COMPS_CSV = "SCD_Complication_Severity_Only.csv"   # person_id, severity_score, binary complication cols
PREF_WIDE = "out/ml_ready_latest.csv"              # <-- new wide file you generated

OUT_DIR   = Path("out_complications")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ========= Load labs (prefer your new wide) =========
labs_path = PREF_WIDE if os.path.exists(PREF_WIDE) else ALT_WIDE
labs = pd.read_csv(labs_path)
comps = pd.read_csv(COMPS_CSV)

# Align person_id type
labs["person_id"] = labs["person_id"].astype(str)
comps["person_id"] = comps["person_id"].astype(str)

# Merge
merged = pd.merge(comps, labs, on="person_id", how="inner")

# Pick lab columns (all non-person_id from the wide file)
lab_columns = [c for c in labs.columns if c != "person_id"]

# Coerce labs to numeric just in case
for c in lab_columns:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

# Identify binary complication columns (exclude id & severity_score)
exclude_cols = {"person_id", "severity_score"}
candidate_comp_cols = [c for c in comps.columns if c not in exclude_cols]

def is_binary_series(s):
    vals = pd.Series(s.dropna().unique())
    return set(vals.tolist()) <= {0, 1}

complication_columns = [c for c in candidate_comp_cols if is_binary_series(merged[c])]

# ========= Helpers =========
def mwus_and_effects(x, y):
    """Mann–Whitney U (two-sided), Cliff’s delta, rank-biserial r."""
    stat, p = mannwhitneyu(x, y, alternative="two-sided", method="auto")
    n1, n2 = len(x), len(y)
    delta = (2 * stat) / (n1 * n2) - 1   # Cliff’s delta
    return stat, p, delta, delta         # (U, p, delta, r_biserial)

def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    mask = ~np.isnan(p)
    if not mask.any():
        return q
    p_valid = p[mask]
    m = p_valid.size
    order = np.argsort(p_valid, kind="mergesort")
    ranks = np.empty_like(order); ranks[order] = np.arange(1, m+1)
    q_valid = p_valid * m / ranks
    # enforce monotonicity
    q_valid = np.minimum.accumulate(q_valid[order[::-1]])[::-1]
    q[mask] = np.minimum(q_valid, 1.0)
    return q

# ========= Run tests =========
rows = []
N_total = merged.shape[0]

for comp in complication_columns:
    yes = merged[comp] == 1
    no  = merged[comp] == 0
    n_yes, n_no = int(yes.sum()), int(no.sum())
    if n_yes < 5 or n_no < 5:
        continue  # skip underpowered splits

    pct_yes = 100.0 * n_yes / N_total
    sev_yes = merged.loc[yes, "severity_score"].median() if "severity_score" in merged.columns else np.nan
    sev_no  = merged.loc[no,  "severity_score"].median() if "severity_score" in merged.columns else np.nan

    for lab in lab_columns:
        g_yes = merged.loc[yes, lab].dropna()
        g_no  = merged.loc[no,  lab].dropna()
        if len(g_yes) < 5 or len(g_no) < 5:
            continue

        med_y, q1_y, q3_y = float(g_yes.median()), float(g_yes.quantile(0.25)), float(g_yes.quantile(0.75))
        med_n, q1_n, q3_n = float(g_no.median()),  float(g_no.quantile(0.25)),  float(g_no.quantile(0.75))

        try:
            U, p, delta, r_rb = mwus_and_effects(g_yes, g_no)
        except Exception:
            U, p, delta, r_rb = np.nan, np.nan, np.nan, np.nan

        rows.append({
            "Complication": comp,
            "Lab Test": lab,
            "n_yes": int(len(g_yes)),
            "n_no": int(len(g_no)),
            "% yes": round(pct_yes, 1),
            "Yes: median [IQR]": f"{med_y:.2f} ({q1_y:.2f}–{q3_y:.2f})",
            "No: median [IQR]":  f"{med_n:.2f} ({q1_n:.2f}–{q3_n:.2f})",
            "Severity median (yes)": sev_yes,
            "Severity median (no)": sev_no,
            "U": U,
            "p_value": p,
            "Cliffs_delta": delta,
            "Rank-biserial r": r_rb
        })

result_df = pd.DataFrame(rows)

# ========= FDR correction & ordering =========
if not result_df.empty:
    result_df["q_value"] = bh_fdr(result_df["p_value"].values)
    result_df = result_df.sort_values(["q_value", "p_value"], na_position="last").reset_index(drop=True)

# ========= Save CSV =========
out_csv = OUT_DIR / "complication_lab_mwu_results.csv"
result_df.to_csv(out_csv, index=False)
print(f"✅ Saved results -> {out_csv} (rows: {len(result_df)})")

# ========= Optional heatmap figure =========
if not result_df.empty:
    mat = result_df.copy()
    mat["q_value"] = mat["q_value"].replace(0, np.nextafter(0, 1))
    mat["signal"] = np.sign(mat["Cliffs_delta"]) * (-np.log10(mat["q_value"]))
    heat = mat.pivot_table(index="Complication", columns="Lab Test", values="signal", aggfunc="mean")
    heat = heat.reindex(index=sorted(heat.index), columns=sorted(heat.columns))

    plt.figure(figsize=(max(8, 0.45*len(heat.columns)), max(6, 0.45*len(heat.index))))
    im = plt.imshow(heat.fillna(0).values, aspect="auto")
    plt.xticks(ticks=np.arange(heat.shape[1]), labels=heat.columns, rotation=75, ha="right")
    plt.yticks(ticks=np.arange(heat.shape[0]), labels=heat.index)
    plt.colorbar(im, label="sign(δ) × -log10(FDR q)")
    plt.title(f"Complication vs Lab (wide source: {os.path.basename(labs_path)})")
    plt.tight_layout()
    fig_path = OUT_DIR / "complication_lab_heatmap.png"
    plt.savefig(fig_path, dpi=300)
    plt.close()
    print(f"🖼️ Saved heatmap -> {fig_path}")

# ========= Peek at top hits =========
with pd.option_context("display.max_rows", 20, "display.max_colwidth", 60):
    print(result_df.head(20))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---- Inputs ----
labs_path = Path("out_complications/complication_lab_mwu_results.csv")

# ---- Load ----
df = pd.read_csv(labs_path)

# ---- Column check ----
needed = {"Complication", "Lab Test", "q_value"}
missing = needed - set(df.columns)
if missing:
    raise KeyError(f"CSV is missing required columns: {missing}")

# If Cliffs_delta missing, try fallback
if "Cliffs_delta" not in df.columns:
    if "median_diff" in df.columns:
        df["Cliffs_delta"] = np.sign(df["median_diff"])
    elif "direction" in df.columns:
        df["Cliffs_delta"] = np.sign(df["direction"])
    else:
        raise KeyError("Need 'Cliffs_delta' (or 'median_diff'/'direction' to infer sign).")

# Replace q=0 to avoid -inf
df["q_value"] = pd.to_numeric(df["q_value"], errors="coerce").replace(0, np.nextafter(0, 1))
df["Cliffs_delta"] = pd.to_numeric(df["Cliffs_delta"], errors="coerce")

# ---- Signal ----
df["signal"] = np.sign(df["Cliffs_delta"]) * (-np.log10(df["q_value"]))

# ---- Pivot ----
heat = df.pivot_table(index="Complication", columns="Lab Test", values="signal", aggfunc="mean")
heat = heat.reindex(index=sorted(heat.index), columns=sorted(heat.columns))

# ---- Plot ----
# ---- Plot ----
plt.figure(figsize=(max(8, 0.45 * heat.shape[1]), max(6, 0.45 * heat.shape[0])))

# Choose a diverging colormap for signed values
im = plt.imshow(heat.fillna(0).values, aspect="auto", cmap="RdBu_r")

plt.xticks(ticks=np.arange(heat.shape[1]), labels=heat.columns, rotation=75, ha="right")
plt.yticks(ticks=np.arange(heat.shape[0]), labels=heat.index)
plt.colorbar(im, label="sign(δ) × -log10(FDR q)")
plt.title(f"Complication vs Lab (source: {os.path.basename(str(labs_path))})")
plt.tight_layout()
plt.show()



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---- Inputs ----
labs_path = Path("out_complications/complication_lab_mwu_results.csv")

# ---- Load ----
df = pd.read_csv(labs_path)

# ---- Prepare data ----
df["q_value"] = pd.to_numeric(df["q_value"], errors="coerce").replace(0, np.nextafter(0, 1))
df["Cliffs_delta"] = pd.to_numeric(df["Cliffs_delta"], errors="coerce")
df["signal"] = np.sign(df["Cliffs_delta"]) * (-np.log10(df["q_value"]))

# Pivot
heat = df.pivot_table(index="Complication", columns="Lab Test", values="signal", aggfunc="mean")
heat = heat.reindex(index=sorted(heat.index), columns=sorted(heat.columns))

# ---- Plot ----
fig, ax = plt.subplots(figsize=(max(8, 0.45*heat.shape[1]), max(6, 0.45*heat.shape[0])))

# Choose a diverging colormap and center at 0
vmax = np.nanmax(np.abs(heat.values))
im = ax.imshow(heat.fillna(0).values, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)

# Axis ticks
ax.set_xticks(np.arange(heat.shape[1]))
ax.set_yticks(np.arange(heat.shape[0]))
ax.set_xticklabels(heat.columns, rotation=75, ha="right")
ax.set_yticklabels(heat.index)

# Annotate each cell
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        if not np.isnan(value):
            ax.text(j, i, f"{value:.1f}", ha="center", va="center", color="black", fontsize=7)

# Colorbar
cbar = ax.figure.colorbar(im, ax=ax)
cbar.set_label("sign(δ) × -log10(FDR q)")

# Title
ax.set_title(f"Complication vs Lab (annotated)\nsource: {os.path.basename(str(labs_path))}")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your MWU results
df = pd.read_csv("out_complications/complication_lab_mwu_results.csv")

# Compute signal
df["q_value"] = df["q_value"].replace(0, np.nextafter(0, 1))
df["signal"] = np.sign(df["Cliffs_delta"]) * (-np.log10(df["q_value"]))

# Pivot table
heat = df.pivot(index="Complication", columns="Lab Test", values="signal")

# Mask non-significant cells (q >= 0.05)
mask = df.pivot(index="Complication", columns="Lab Test", values="q_value") >= 0.05

# Sort rows and columns for clustering
heat = heat.loc[sorted(heat.index), sorted(heat.columns)]
mask = mask.loc[heat.index, heat.columns]

# Plot with seaborn
plt.figure(figsize=(12, 6))
sns.heatmap(
    heat,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    annot=True, fmt=".1f",  # show values
    linewidths=0.5,
    cbar_kws={"label": "sign(δ) × -log10(FDR q)"}
)
plt.title("Complication vs Lab Test (significant associations only)")
plt.tight_layout()
plt.show()


---
## Step 12 — Backup Results to Workspace Bucket

**What this does:**  
Copies all output CSV files and figures to your All of Us workspace
Google Cloud Storage bucket for safe keeping.

> 💾 This ensures your results are preserved even if your Jupyter environment resets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load your results
df = pd.read_csv("out_complications/complication_lab_mwu_results.csv")

# Clean names (trailing spaces etc.)
df["Complication"] = df["Complication"].astype(str).str.strip()
df["Lab Test"]     = df["Lab Test"].astype(str).str.strip()

# --- DEFINE THE FULL SET YOU WANT TO SEE ---
# Option A: show ALL complications from your master list (replace with yours):
full_complications = [
    "Pulmonary hypertension","Vitreous haemorrhage","Liver cirrhosis","Osteomyelitis",
    "Preeclampsia","Aplastic crisis / Aplastic Anemia","Hand and foot syndrome",
    "Sickle retinopathy","Restrictive lung disease","Dactylitis",
    # ... add the rest you want visible (e.g., your whole SCD list or your computed Top 10)
]

# Option B: or if you already have top10 as a DataFrame:
# full_complications = top10["Complication"].tolist()

# Labs: include ALL labs that appear anywhere in the file
full_labs = sorted(df["Lab Test"].unique().tolist())

# Make categories so pivot keeps empty rows/cols
df["Complication"] = pd.Categorical(df["Complication"], categories=full_complications, ordered=True)
df["Lab Test"]     = pd.Categorical(df["Lab Test"],     categories=full_labs,         ordered=True)

# Build signal
df["q_value"] = pd.to_numeric(df["q_value"], errors="coerce").replace(0, np.nextafter(0, 1))
if "Cliffs_delta" not in df:
    raise KeyError("Need 'Cliffs_delta' to compute sign; add it or provide a fallback.")
df["Cliffs_delta"] = pd.to_numeric(df["Cliffs_delta"], errors="coerce")
df["signal"] = np.sign(df["Cliffs_delta"]) * (-np.log10(df["q_value"]))

# Optional: zero out non-significant
# df.loc[df["q_value"] >= 0.05, "signal"] = 0

# Pivot (observed=False keeps all categorical levels)
heat = df.pivot_table(
    index="Complication",
    columns="Lab Test",
    values="signal",
    aggfunc="mean",
    observed=False
)

# Ensure full grid/order and fill NaNs with 0 so they still draw
heat = heat.reindex(index=full_complications, columns=full_labs).fillna(0)

# Plot with color
plt.figure(figsize=(max(8, 0.45*heat.shape[1]), max(6, 0.45*heat.shape[0])))
im = plt.imshow(heat.values, aspect="auto", cmap="RdBu_r")
plt.xticks(range(heat.shape[1]), heat.columns, rotation=75, ha="right")
plt.yticks(range(heat.shape[0]), heat.index)
cbar = plt.colorbar(im, label="sign(δ) × -log10(FDR q)")
# Center the colormap at 0 for diverging data:
vmax = np.nanmax(np.abs(heat.values))
im.set_clim(-vmax, vmax)
plt.title("Complication vs Lab (forced full list)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ===================== INPUTS =====================
MWU_PATH   = Path("out_complications/complication_lab_mwu_results.csv")  # your MWU results
PREV_PATH  = Path("scd_complication_counts_all.csv")                     # has Complication, n_patients, pct_of_476
N_PARTICIPANTS = 476                                                     # denominator for prevalence (%)

# Show only the top-N most prevalent complications (set to None for all)
SHOW_TOP_N = 10

# Only annotate significant cells?
ANNOTATE_SIGNIFICANT_ONLY = True
Q_SIG_THRESHOLD = 0.05

# Save figure? (set to None to not save)
SAVE_PNG = "complication_lab_heatmap_with_prevalence.png"
# ==================================================

# ---------- Load MWU results ----------
df = pd.read_csv(MWU_PATH)

# Ensure numeric and avoid -inf from log(0)
df["q_value"] = pd.to_numeric(df["q_value"], errors="coerce").replace(0, np.nextafter(0, 1))
df["Cliffs_delta"] = pd.to_numeric(df["Cliffs_delta"], errors="coerce")

# Build signal = sign(δ) × −log10(q)
df["signal"] = np.sign(df["Cliffs_delta"]) * (-np.log10(df["q_value"]))

# Clean names
df["Complication"] = df["Complication"].astype(str).str.strip()
df["Lab Test"]     = df["Lab Test"].astype(str).str.strip()

# ---------- Load prevalence (for labels) ----------
prev = pd.read_csv(PREV_PATH)
# If your file doesn't already have pct_of_476, compute it:
if "pct_of_476" not in prev.columns:
    prev["pct_of_476"] = (prev["n_patients"] / float(N_PARTICIPANTS) * 100).round(1)

prev = prev[["Complication", "pct_of_476", "n_patients"]].drop_duplicates()
prev_map_pct = prev.set_index("Complication")["pct_of_476"].to_dict()

# Optionally pick top-N by prevalence
if SHOW_TOP_N is not None:
    topN_list = (prev.sort_values(["n_patients", "Complication"], ascending=[False, True])
                     .head(SHOW_TOP_N)["Complication"].tolist())
    df = df[df["Complication"].isin(topN_list)]

# ---------- Build matrices ----------
# signal matrix
heat = df.pivot_table(index="Complication", columns="Lab Test", values="signal", aggfunc="mean")

# q-value matrix (for significance mask & optional annotation)
qmat = df.pivot_table(index="Complication", columns="Lab Test", values="q_value", aggfunc="min")

# sort rows/cols for readability
heat = heat.reindex(index=sorted(heat.index), columns=sorted(heat.columns))
qmat = qmat.reindex(index=heat.index, columns=heat.columns)

# ---------- Prepare masked array (grey = no data) ----------
data = heat.values.astype(float)
mask = np.isnan(data)
data_ma = np.ma.masked_array(data, mask=mask)

cmap = plt.cm.RdBu_r
cmap.set_bad("#eeeeee")  # light grey for missing/no-data cells

# Symmetric color scale centered at 0
vmax = np.nanmax(np.abs(data))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0  # fallback to avoid errors if all zeros/NaNs
vmin = -vmax

# ---------- Plot ----------
fig_h = max(6, 0.5 * heat.shape[0])
fig_w = max(8, 0.5 * heat.shape[1])
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

im = ax.imshow(data_ma, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

# Axis ticks & labels (append prevalence % to y labels)
ax.set_xticks(np.arange(heat.shape[1]))
ax.set_yticks(np.arange(heat.shape[0]))
ax.set_xticklabels(heat.columns, rotation=75, ha="right")

y_labels = []
for comp in heat.index:
    pct = prev_map_pct.get(comp, np.nan)
    if np.isfinite(pct):
        y_labels.append(f"{comp} ({pct:.1f}%)")
    else:
        y_labels.append(comp)
ax.set_yticklabels(y_labels)

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("sign(δ) × -log10(FDR q)")

# ---------- Annotate cells ----------
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iat[i, j]
        if np.isnan(val):
            continue  # no data
        if ANNOTATE_SIGNIFICANT_ONLY:
            qv = qmat.iat[i, j]
            if pd.isna(qv) or qv >= Q_SIG_THRESHOLD:
                continue  # skip non-significant
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7, color="black")

ax.set_title(f"Complication vs Lab (with prevalence)\nsource: {MWU_PATH.name}")
plt.tight_layout()

if SAVE_PNG:
    plt.savefig(SAVE_PNG, dpi=300)
plt.show()


In [ ]:
import shutil
import os

files_to_save = [
    "scd_complication_counts_all.csv",
    "scd_complication_top10.csv",
    "SCD_Complication_Severity_Categorised.csv",
    "SCD_Complication_Severity_Only.csv",
    "SCD_Demographics.csv",
    "SCD_Demographics_with_Complications.csv",
    "your_condition_occurrence_data.csv",
    "complication_lab_heatmap_with_prevalence.png",
    "complication_lab_heatmap.png",
    "complication_lab_mwu_results.csv"
]

# Make save folder
save_folder = "SCD_outputs"
os.makedirs(save_folder, exist_ok=True)

# Copy files
for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, save_folder)

# Zip the folder
shutil.make_archive("SCD_outputs", 'zip', save_folder)
print("Zipped outputs to SCD_outputs.zip — ready to download.")


---
## ✅ Notebook 03 Complete — Pipeline Finished!

**Output files created:**
| File | Description |
|------|-------------|
| `SCD_Lab_Measurements_Long.csv` | Long-format raw lab data |
| `lab_measurements_wide_with_units.csv` | Wide-format lab data (per patient) |
| `ml_ready_latest.csv` | ML-ready feature matrix |
| `out_complications/complication_lab_mwu_results.csv` | MWU statistical results |
| `out/summary_medians.csv` | Cohort median values per analyte |

**The full SCD multi-modal extraction pipeline is complete.**

---
> 💡 **Next steps for analysis:**
> - Use `ml_ready_latest.csv` for supervised ML (e.g. severity prediction)
> - Use `complication_lab_mwu_results.csv` for publication-ready statistical tables
> - All figures are saved in `./out/` and `./out_complications/`